# 3 — Construcción de Gold

Gold publica tres productos con granos explícitos: ventas diarias, riesgo por cliente y resumen operativo por lote. Se reconstruyen desde Silver para que una reejecución sea determinista.

In [0]:
dbutils.widgets.text("student_id", "alumno01")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"])
dbutils.widgets.text("expected_batch_id", "batch_002")
dbutils.widgets.text("job_run_id", "interactive")

In [0]:
%run ../common/config

In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
tx = qualified_table(config, "silver_transactions")
customers = qualified_table(config, "silver_customers")
products = qualified_table(config, "silver_products")
quarantine = qualified_table(config, "silver_transactions_quarantine")
for name in [tx, customers, products, quarantine]:
    assert spark.catalog.tableExists(name), f"Falta {name}. Ejecutá la tarea 02."

In [0]:
daily = qualified_table(config, "gold_daily_sales")
spark.sql(f"""
CREATE OR REPLACE TABLE {daily} USING DELTA AS
SELECT to_date(t.event_ts) sale_date, c.country, p.category, t.payment_channel,
       count(*) transaction_count, count(DISTINCT t.customer_id) unique_customers,
       cast(sum(t.amount) AS DECIMAL(18,2)) total_amount,
       cast(avg(t.amount) AS DECIMAL(18,2)) average_amount,
       sum(t.is_fraud) fraud_transactions,
       cast(sum(t.is_fraud) / count(*) AS DECIMAL(8,4)) fraud_rate
FROM {tx} t
JOIN {customers} c ON t.customer_id = c.customer_id
JOIN {products} p ON t.product_id = p.product_id
GROUP BY to_date(t.event_ts), c.country, p.category, t.payment_channel
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
risk = qualified_table(config, "gold_customer_risk")
spark.sql(f"""
CREATE OR REPLACE TABLE {risk} USING DELTA AS
WITH metrics AS (
  SELECT t.customer_id, c.country, c.segment, count(*) transaction_count,
         cast(sum(t.amount) AS DECIMAL(18,2)) total_amount,
         cast(avg(t.amount) AS DECIMAL(18,2)) average_amount,
         count(DISTINCT t.device_id) distinct_devices, sum(t.is_fraud) fraud_transactions,
         max(t.event_ts) last_transaction_at
  FROM {tx} t JOIN {customers} c ON t.customer_id = c.customer_id
  GROUP BY t.customer_id, c.country, c.segment
)
SELECT *, CASE WHEN fraud_transactions >= 2 THEN 'high'
               WHEN fraud_transactions = 1 OR average_amount >= 1500 THEN 'medium'
               ELSE 'low' END risk_level
FROM metrics
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
batch = qualified_table(config, "gold_batch_summary")
spark.sql(f"""
CREATE OR REPLACE TABLE {batch} USING DELTA AS
WITH accepted AS (
  SELECT source_batch_id, count(*) accepted_transactions,
         cast(sum(amount) AS DECIMAL(18,2)) total_amount, sum(is_fraud) fraud_transactions
  FROM {tx} GROUP BY source_batch_id
), rejected AS (
  SELECT source_batch_id, count(*) rejected_transactions FROM {quarantine} GROUP BY source_batch_id
)
SELECT coalesce(a.source_batch_id, r.source_batch_id) source_batch_id,
       coalesce(a.accepted_transactions, 0) accepted_transactions,
       coalesce(r.rejected_transactions, 0) rejected_transactions,
       coalesce(a.total_amount, cast(0 AS DECIMAL(18,2))) total_amount,
       coalesce(a.fraud_transactions, 0) fraud_transactions, current_timestamp() refreshed_at
FROM accepted a FULL OUTER JOIN rejected r ON a.source_batch_id = r.source_batch_id
""")
display(spark.table(batch).orderBy("source_batch_id"))
display(spark.table(daily).orderBy("sale_date", "country").limit(20))

source_batch_id,accepted_transactions,rejected_transactions,total_amount,fraud_transactions,refreshed_at
batch_002,200,2,144813.00,27,2026-10-05T21:43:41.960Z
batch_003,201,2,147014.99,28,2026-10-05T21:43:41.960Z
initial,49948,51,50139370.34,6560,2026-10-05T21:43:41.960Z


sale_date,country,category,payment_channel,transaction_count,unique_customers,total_amount,average_amount,fraud_transactions,fraud_rate
2026-03-01,AR,electronics,wallet,89,85,92875.15,1043.54,13,0.1461
2026-03-01,AR,electronics,card,91,90,94998.64,1043.94,13,0.1429
2026-03-01,AR,books,transfer,107,101,118050.23,1103.27,17,0.1589
2026-03-01,AR,sports,card,85,84,85331.33,1003.90,10,0.1176
2026-03-01,AR,fashion,card,97,95,85756.84,884.09,10,0.1031
2026-03-01,AR,fashion,transfer,83,81,74316.83,895.38,9,0.1084
2026-03-01,AR,fashion,wallet,76,74,75204.89,989.54,13,0.1711
2026-03-01,AR,books,wallet,117,111,111622.66,954.04,14,0.1197
2026-03-01,AR,home,wallet,118,111,123044.16,1042.75,14,0.1186
2026-03-01,AR,books,card,102,100,113590.85,1113.64,14,0.1373


In [0]:
from pyspark.sql import functions as F
resultado_6 = (
    spark.table(batch)
        .withColumn("total", F.col("accepted_transactions") + F.col("rejected_transactions"))
        .withColumn(
          "tasa_rechazo",
          F.when(F.col("total") == 0, None)
           .otherwise(F.col("rejected_transactions") / F.col("total"))
      )
      .orderBy(F.col("tasa_rechazo").desc())
)

display(resultado_6)

source_batch_id,accepted_transactions,rejected_transactions,total_amount,fraud_transactions,refreshed_at,total,tasa_rechazo
batch_002,200,2,144813.00,27,2026-10-05T21:43:41.960Z,202,0.009900990099009901
batch_003,201,2,147014.99,28,2026-10-05T21:43:41.960Z,203,0.009852216748768473
initial,49948,51,50139370.34,6560,2026-10-05T21:43:41.960Z,49999,0.0010200204004080081


In [0]:
resultado_7 = (
    spark.table(daily)
      .groupBy("sale_date")
      .agg(
          F.sum("total_amount").alias("monto_total"),
          F.sum("transaction_count").alias("cantidad_transacciones")
      )
)

max_monto = resultado_7.agg(F.max("monto_total")).first()[0]
max_trx = resultado_7.agg(F.max("cantidad_transacciones")).first()[0]

display(resultado_7)
print("Día(s) con mayor monto total:")
display(resultado_7.filter(F.col("monto_total") == max_monto))
print("Día(s) con mayor cantidad de transacciones:")
display(resultado_7.filter(F.col("cantidad_transacciones") == max_trx))

sale_date,monto_total,cantidad_transacciones
2026-03-01,7310840.77,7236
2026-03-02,7271589.97,7183
2026-03-07,7089104.28,7165
2026-03-03,7167008.24,7127
2026-03-04,7091526.73,7150
2026-03-05,7010989.71,6958
2026-03-06,7198310.64,7129
2026-03-10,144813.00,200
2026-03-11,147014.99,201


Día(s) con mayor monto total:


sale_date,monto_total,cantidad_transacciones
2026-03-01,7310840.77,7236


Día(s) con mayor cantidad de transacciones:


sale_date,monto_total,cantidad_transacciones
2026-03-01,7310840.77,7236


In [0]:
resultado_8 = (
    spark.table(daily)
    .groupBy("payment_channel")
      .agg(
          F.sum("fraud_transactions").alias("total_fraude"),
          F.sum("transaction_count").alias("total_transacciones"),
      )
      .withColumn(
          "tasa_fraude_global",
          F.col("total_fraude") / F.col("total_transacciones")
      )
      .orderBy(F.col("tasa_fraude_global").desc())
)

display(resultado_8)

payment_channel,total_fraude,total_transacciones,tasa_fraude_global
transfer,2327,16723,0.1391496741015368
wallet,2169,16724,0.1296938531451806
card,2119,16902,0.12536977872441132


In [0]:
from pyspark.sql import functions as F, Window
resultado_9 = (spark.table(daily)
               .groupBy("country","category")
               .agg(F.sum("total_amount").alias("monto_total"))
               )

top_global = resultado_9.orderBy(F.col("monto_total").desc()).limit(1)
display(top_global)

w = Window.partitionBy("country").orderBy(F.col("monto_total").desc())

top_por_pais = (
    resultado_9
      .withColumn("rn", F.row_number().over(w))
      .filter(F.col("rn") == 1)
      .drop("rn")
      .orderBy(F.col("monto_total").desc())
)
display(top_por_pais)

country,category,monto_total
BR,home,2361959.97


country,category,monto_total
BR,home,2361959.97
UY,home,2323924.46
AR,books,2267722.26
MX,books,2235941.63
CL,home,2167748.03
